## LangGraph Open Deep Research - Supervisor-Researcher Architecture

In this notebook, we'll explore the **supervisor-researcher delegation architecture** for conducting deep research with LangGraph.

You can visit this repository to see the original application: [Open Deep Research](https://github.com/langchain-ai/open_deep_research)

Let's jump in!

## What We're Building

This implementation uses a **hierarchical delegation pattern** where:

1. **User Clarification** - Optionally asks clarifying questions to understand the research scope
2. **Research Brief Generation** - Transforms user messages into a structured research brief
3. **Supervisor** - A lead researcher that analyzes the brief and delegates research tasks
4. **Parallel Researchers** - Multiple sub-agents that conduct focused research simultaneously
5. **Research Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings are combined into a comprehensive report

![Architecture Diagram](https://i.imgur.com/Q8HEZn0.png)

This differs from a section-based approach by allowing dynamic task decomposition based on the research question, rather than predefined sections.

---

# 🤝 Breakout Room #1
## Deep Research Foundations

In this breakout room, we'll understand the architecture and components of the Open Deep Research system.

## Task 1: Dependencies

You'll need API keys for Anthropic (for the LLM) and Tavily (for web search). We'll configure the system to use Anthropic's Claude Sonnet 4 exclusively.

In [1]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter your Tavily API key: ")

## Task 2: State Definitions

The state structure is hierarchical with three levels:

### Agent State (Top Level)
Contains the overall conversation messages, research brief, accumulated notes, and final report.

### Supervisor State (Middle Level)
Manages the research supervisor's messages, research iterations, and coordinating parallel researchers.

### Researcher State (Bottom Level)
Each individual researcher has their own message history, tool call iterations, and research findings.

We also have structured outputs for tool calling:
- **ConductResearch** - Tool for supervisor to delegate research to a sub-agent
- **ResearchComplete** - Tool to signal research phase is done
- **ClarifyWithUser** - Structured output for asking clarifying questions
- **ResearchQuestion** - Structured output for the research brief

Let's import these from our library: [`open_deep_library/state.py`](open_deep_library/state.py)

In [2]:
# Import state definitions from the library
from open_deep_library.state import (
    # Main workflow states
    AgentState,           # Lines 65-72: Top-level agent state with messages, research_brief, notes, final_report
    AgentInputState,      # Lines 62-63: Input state is just messages
    
    # Supervisor states
    SupervisorState,      # Lines 74-81: Supervisor manages research delegation and iterations
    
    # Researcher states
    ResearcherState,      # Lines 83-90: Individual researcher with messages and tool iterations
    ResearcherOutputState, # Lines 92-96: Output from researcher (compressed research + raw notes)
    
    # Structured outputs for tool calling
    ConductResearch,      # Lines 15-19: Tool for delegating research to sub-agents
    ResearchComplete,     # Lines 21-22: Tool to signal research completion
    ClarifyWithUser,      # Lines 30-41: Structured output for user clarification
    ResearchQuestion,     # Lines 43-48: Structured output for research brief
)

## Task 3: Utility Functions and Tools

The system uses several key utilities:

### Search Tools
- **tavily_search** - Async web search with automatic summarization to stay within token limits
- Supports Anthropic native web search and Tavily API

### Reflection Tools
- **think_tool** - Allows researchers to reflect on their progress and plan next steps (ReAct pattern)

### Helper Utilities
- **get_all_tools** - Assembles the complete toolkit (search + MCP + reflection)
- **get_today_str** - Provides current date context for research
- Token limit handling utilities for graceful degradation

These are defined in [`open_deep_library/utils.py`](open_deep_library/utils.py)

In [3]:
# Import utility functions and tools from the library
from open_deep_library.utils import (
    # Search tool - Lines 43-136: Tavily search with automatic summarization
    tavily_search,
    
    # Reflection tool - Lines 219-244: Strategic thinking tool for ReAct pattern
    think_tool,
    
    # Tool assembly - Lines 569-597: Get all configured tools
    get_all_tools,
    
    # Date utility - Lines 872-879: Get formatted current date
    get_today_str,
    
    # Supporting utilities for error handling
    get_api_key_for_model,          # Lines 892-914: Get API keys from config or env
    is_token_limit_exceeded,         # Lines 665-701: Detect token limit errors
    get_model_token_limit,           # Lines 831-846: Look up model's token limit
    remove_up_to_last_ai_message,    # Lines 848-866: Truncate messages for retry
    anthropic_websearch_called,      # Lines 607-637: Detect Anthropic native search usage
    openai_websearch_called,         # Lines 639-658: Detect OpenAI native search usage
    get_notes_from_tool_calls,       # Lines 599-601: Extract notes from tool messages
)

## Task 4: Configuration System

The configuration system controls:

### Research Behavior
- **allow_clarification** - Whether to ask clarifying questions before research
- **max_concurrent_research_units** - How many parallel researchers can run (default: 5)
- **max_researcher_iterations** - How many times supervisor can delegate research (default: 6)
- **max_react_tool_calls** - Tool call limit per researcher (default: 10)

### Model Configuration
- **research_model** - Model for research and supervision (we'll use Anthropic)
- **compression_model** - Model for synthesizing findings
- **final_report_model** - Model for writing the final report
- **summarization_model** - Model for summarizing web search results

### Search Configuration
- **search_api** - Which search API to use (ANTHROPIC, TAVILY, or NONE)
- **max_content_length** - Character limit before summarization

Defined in [`open_deep_library/configuration.py`](open_deep_library/configuration.py)

In [4]:
# Import configuration from the library
from open_deep_library.configuration import (
    Configuration,    # Lines 38-247: Main configuration class with all settings
    SearchAPI,        # Lines 11-17: Enum for search API options (ANTHROPIC, TAVILY, NONE)
)

## Task 5: Prompt Templates

The system uses carefully engineered prompts for each phase:

### Phase 1: Clarification
**clarify_with_user_instructions** - Analyzes if the research scope is clear or needs clarification

### Phase 2: Research Brief
**transform_messages_into_research_topic_prompt** - Converts user messages into a detailed research brief

### Phase 3: Supervisor
**lead_researcher_prompt** - System prompt for the supervisor that manages delegation strategy

### Phase 4: Researcher
**research_system_prompt** - System prompt for individual researchers conducting focused research

### Phase 5: Compression
**compress_research_system_prompt** - Prompt for synthesizing research findings without losing information

### Phase 6: Final Report
**final_report_generation_prompt** - Comprehensive prompt for writing the final report

All prompts are defined in [`open_deep_library/prompts.py`](open_deep_library/prompts.py)

In [5]:
# Import prompt templates from the library
from open_deep_library.prompts import (
    clarify_with_user_instructions,                    # Lines 3-41: Ask clarifying questions
    transform_messages_into_research_topic_prompt,     # Lines 44-77: Generate research brief
    lead_researcher_prompt,                            # Lines 79-136: Supervisor system prompt
    research_system_prompt,                            # Lines 138-183: Researcher system prompt
    compress_research_system_prompt,                   # Lines 186-222: Research compression prompt
    final_report_generation_prompt,                    # Lines 228-308: Final report generation
)

## ❓ Question #1:

Explain the interrelationships between the three states (Agent, Supervisor, Researcher). Why don't we just make a single huge state?

##### Answer:
The three states are intentionally layered so each plays a different role: the Agent state holds the overall conversation, the research brief, aggregated notes and the final report. The Supervisor state lives inside the agent workflow as the manager that plans, applies budgets/limits, delegates ConductResearch calls, and collects tool results and raw notes from parallel work—it enforces iteration and concurrency rules before handing results back up. Individual Researcher states are per-subagent, short-lived contexts used to run focused toolccalling loops, gather observations, and produce compressed outputs that the supervisor aggregates. Each researcher keeps its own messages, iterations and compressed_research. Putting everything into one state would bloat prompts and token usage, make parallel execution and error isolation hard, and mix distinct permission/security/tool scopes. The supervisor’s delegation and the researcher’s tool loops are simpler to reason about when their states are separate.

## ❓ Question #2:

What are the advantages and disadvantages of importing these components instead of including them in the notebook?

##### Answer:
Importing components from separate files brings clearer engineering benefits but costs some immediacy and transparency compared with embedding everything in the notebook. When we import, the codebase becomes modular: each file can be versioned, unit-tested, and reused across projects or multiple notebooks which improves readability. On the other hand, we need many other files to be able to read and run the notebook itself which kind of looses the sense of making one it the first place.

## 🏗️ Activity #1: Explore the Prompts

Open `open_deep_library/prompts.py` and examine one of the prompt templates in detail.

**Requirements:**
1. Choose one prompt template (clarify, brief, supervisor, researcher, compression, or final report)
2. Explain what the prompt is designed to accomplish
3. Identify 2-3 key techniques used in the prompt (e.g., structured output, role definition, examples)
4. Suggest one improvement you might make to the prompt

**YOUR CODE HERE** - Write your analysis in a markdown cell below

I chose the research supervisor prompt.

This prompt is designed to make the agent act as a research supervisor whose job is to orchestrate focused research by repeatedly delegating sub-tasks to specialized researcher subagents (via ConductResearch), reflecting on progress (think_tool), and stopping when enough evidence has been gathered (ResearchComplete). 
The prompt uses a few effective techniques. First, it defines a clear role and goal up front (“You are a research supervisor”) so the model’s behavior is anchored to a single persona. Second, it enforces a structured workflow by explicitly telling the agent which tools to use and when (think → conduct → reflect → stop), which reduces ambiguity and encourages repeatable behavior. Third, it uses operational constraints and examples (budgets, max iterations, scaling rules, and short examples) to encode trade-offs so the supervisor makes pragmatic delegation choices rather than chasing exhaustive research.
One improvement would be to add a short, structured summary (for example in JSON format) after each planning or delegation step, instead of only free-form reasoning - for example: {"plan":"...", "delegation_targets":[...], "confidence":0.0, "missing_items":[...]}. This improves determinism and reliability. If every iteration produces the same structured fields, the system can automatically check whether required reasoning steps were completed and whether the agent’s confidence justifies stopping. Also, it enhances observability and debugging. Structured summaries make it easier to trace why the supervisor decided to delegate more research or to stop, rather than having to interpret long free-text reflections.

---

# 🤝 Breakout Room #2
## Building & Running the Researcher

In this breakout room, we'll explore the node functions, build the graph, and run wellness research.

## Task 6: Node Functions - The Building Blocks

Now let's look at the node functions that make up our graph. We'll import them from the library and understand what each does.

### The Complete Research Workflow

The workflow consists of 8 key nodes organized into 3 subgraphs:

1. **Main Graph Nodes:**
   - `clarify_with_user` - Entry point that checks if clarification is needed
   - `write_research_brief` - Transforms user input into structured research brief
   - `final_report_generation` - Synthesizes all research into final report

2. **Supervisor Subgraph Nodes:**
   - `supervisor` - Lead researcher that plans and delegates
   - `supervisor_tools` - Executes supervisor's tool calls (delegation, reflection)

3. **Researcher Subgraph Nodes:**
   - `researcher` - Individual researcher conducting focused research
   - `researcher_tools` - Executes researcher's tool calls (search, reflection)
   - `compress_research` - Synthesizes researcher's findings

All nodes are defined in [`open_deep_library/deep_researcher.py`](open_deep_library/deep_researcher.py)

### Node 1: clarify_with_user

**Purpose:** Analyzes user messages and asks clarifying questions if the research scope is unclear.

**Key Steps:**
1. Check if clarification is enabled in configuration
2. Use structured output to analyze if clarification is needed
3. If needed, end with a clarifying question for the user
4. If not needed, proceed to research brief with verification message

**Implementation:** [`open_deep_library/deep_researcher.py` lines 60-115](open_deep_library/deep_researcher.py#L60-L115)

In [6]:
# Import the clarify_with_user node
from open_deep_library.deep_researcher import clarify_with_user

### Node 2: write_research_brief

**Purpose:** Transforms user messages into a structured research brief for the supervisor.

**Key Steps:**
1. Use structured output to generate detailed research brief from messages
2. Initialize supervisor with system prompt and research brief
3. Set up supervisor messages with proper context

**Why this matters:** A well-structured research brief helps the supervisor make better delegation decisions.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 118-175](open_deep_library/deep_researcher.py#L118-L175)

In [7]:
# Import the write_research_brief node
from open_deep_library.deep_researcher import write_research_brief

### Node 3: supervisor

**Purpose:** Lead research supervisor that plans research strategy and delegates to sub-researchers.

**Key Steps:**
1. Configure model with three tools:
   - `ConductResearch` - Delegate research to a sub-agent
   - `ResearchComplete` - Signal that research is done
   - `think_tool` - Strategic reflection before decisions
2. Generate response based on current context
3. Increment research iteration count
4. Proceed to tool execution

**Decision Making:** The supervisor uses `think_tool` to reflect before delegating research, ensuring thoughtful decomposition of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 178-223](open_deep_library/deep_researcher.py#L178-L223)

In [8]:
# Import the supervisor node (from supervisor subgraph)
from open_deep_library.deep_researcher import supervisor

### Node 4: supervisor_tools

**Purpose:** Executes the supervisor's tool calls, including strategic thinking and research delegation.

**Key Steps:**
1. Check exit conditions:
   - Exceeded maximum iterations
   - No tool calls made
   - `ResearchComplete` called
2. Process `think_tool` calls for strategic reflection
3. Execute `ConductResearch` calls in parallel:
   - Spawn researcher subgraphs for each delegation
   - Limit to `max_concurrent_research_units` (default: 5)
   - Gather all results asynchronously
4. Aggregate findings and return to supervisor

**Parallel Execution:** This is where the magic happens - multiple researchers work simultaneously on different aspects of the research question.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 225-349](open_deep_library/deep_researcher.py#L225-L349)

In [9]:
# Import the supervisor_tools node
from open_deep_library.deep_researcher import supervisor_tools

### Node 5: researcher

**Purpose:** Individual researcher that conducts focused research on a specific topic.

**Key Steps:**
1. Load all available tools (search, MCP, reflection)
2. Configure model with tools and researcher system prompt
3. Generate response with tool calls
4. Increment tool call iteration count

**ReAct Pattern:** Researchers use `think_tool` to reflect after each search, deciding whether to continue or provide their answer.

**Available Tools:**
- Search tools (Tavily or Anthropic native search)
- `think_tool` for strategic reflection
- `ResearchComplete` to signal completion
- MCP tools (if configured)

**Implementation:** [`open_deep_library/deep_researcher.py` lines 365-424](open_deep_library/deep_researcher.py#L365-L424)

In [10]:
# Import the researcher node (from researcher subgraph)
from open_deep_library.deep_researcher import researcher

### Node 6: researcher_tools

**Purpose:** Executes the researcher's tool calls, including searches and strategic reflection.

**Key Steps:**
1. Check early exit conditions (no tool calls, native search used)
2. Execute all tool calls in parallel:
   - Search tools fetch and summarize web content
   - `think_tool` records strategic reflections
   - MCP tools execute external integrations
3. Check late exit conditions:
   - Exceeded `max_react_tool_calls` (default: 10)
   - `ResearchComplete` called
4. Continue research loop or proceed to compression

**Error Handling:** Safely handles tool execution errors and continues with available results.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 435-509](open_deep_library/deep_researcher.py#L435-L509)

In [11]:
# Import the researcher_tools node
from open_deep_library.deep_researcher import researcher_tools

### Node 7: compress_research

**Purpose:** Compresses and synthesizes research findings into a concise, structured summary.

**Key Steps:**
1. Configure compression model
2. Add compression instruction to messages
3. Attempt compression with retry logic:
   - If token limit exceeded, remove older messages
   - Retry up to 3 times
4. Extract raw notes from tool and AI messages
5. Return compressed research and raw notes

**Why Compression?** Researchers may accumulate lots of tool outputs and reflections. Compression ensures:
- All important information is preserved
- Redundant information is deduplicated
- Content stays within token limits for the final report

**Token Limit Handling:** Gracefully handles token limit errors by progressively truncating messages.

**Implementation:** [`open_deep_library/deep_researcher.py` lines 511-585](open_deep_library/deep_researcher.py#L511-L585)

In [12]:
# Import the compress_research node
from open_deep_library.deep_researcher import compress_research

### Node 8: final_report_generation

**Purpose:** Generates the final comprehensive research report from all collected findings.

**Key Steps:**
1. Extract all notes from completed research
2. Configure final report model
3. Attempt report generation with retry logic:
   - If token limit exceeded, truncate findings by 10%
   - Retry up to 3 times
4. Return final report or error message

**Token Limit Strategy:**
- First retry: Use model's token limit × 4 as character limit
- Subsequent retries: Reduce by 10% each time
- Graceful degradation with helpful error messages

**Report Quality:** The prompt guides the model to create well-structured reports with:
- Proper headings and sections
- Inline citations
- Comprehensive coverage of all findings
- Sources section at the end

**Implementation:** [`open_deep_library/deep_researcher.py` lines 607-697](open_deep_library/deep_researcher.py#L607-L697)

In [13]:
# Import the final_report_generation node
from open_deep_library.deep_researcher import final_report_generation

## Task 7: Graph Construction - Putting It All Together

The system is organized into three interconnected graphs:

### 1. Researcher Subgraph (Bottom Level)
Handles individual focused research on a specific topic:
```
START → researcher → researcher_tools → compress_research → END
               ↑            ↓
               └────────────┘ (loops until max iterations or ResearchComplete)
```

### 2. Supervisor Subgraph (Middle Level)
Manages research delegation and coordination:
```
START → supervisor → supervisor_tools → END
            ↑              ↓
            └──────────────┘ (loops until max iterations or ResearchComplete)
            
supervisor_tools spawns multiple researcher_subgraphs in parallel
```

### 3. Main Deep Researcher Graph (Top Level)
Orchestrates the complete research workflow:
```
START → clarify_with_user → write_research_brief → research_supervisor → final_report_generation → END
                 ↓                                       (supervisor_subgraph)
               (may end early if clarification needed)
```

Let's import the compiled graphs from the library.

In [14]:
# Import the pre-compiled graphs from the library
from open_deep_library.deep_researcher import (
    # Bottom level: Individual researcher workflow
    researcher_subgraph,    # Lines 588-605: researcher → researcher_tools → compress_research
    
    # Middle level: Supervisor coordination
    supervisor_subgraph,    # Lines 351-363: supervisor → supervisor_tools (spawns researchers)
    
    # Top level: Complete research workflow
    deep_researcher,        # Lines 699-719: Main graph with all phases
)

## Why This Architecture?

### Advantages of Supervisor-Researcher Delegation

1. **Dynamic Task Decomposition**
   - Unlike section-based approaches with predefined structure, the supervisor can break down research based on the actual question
   - Adapts to different types of research (comparisons, lists, deep dives, etc.)

2. **Parallel Execution**
   - Multiple researchers work simultaneously on different aspects
   - Much faster than sequential section processing
   - Configurable parallelism (1-20 concurrent researchers)

3. **ReAct Pattern for Quality**
   - Researchers use `think_tool` to reflect after each search
   - Prevents excessive searching and improves search quality
   - Natural stopping conditions based on information sufficiency

4. **Flexible Tool Integration**
   - Easy to add MCP tools for specialized research
   - Supports multiple search APIs (Anthropic, Tavily)
   - Each researcher can use different tool combinations

5. **Graceful Token Limit Handling**
   - Compression prevents token overflow
   - Progressive truncation in final report generation
   - Research can scale to arbitrary depths

### Trade-offs

- **Complexity:** More moving parts than section-based approach
- **Cost:** Parallel researchers use more tokens (but faster)
- **Unpredictability:** Research structure emerges dynamically

## Task 8: Running the Deep Researcher

Now let's see the system in action! We'll use it to research wellness strategies for improving sleep quality.

### Setup

We need to:
1. Set up the wellness research request
2. Configure the execution with Anthropic settings
3. Run the research workflow

In [15]:
# Set up the graph with Anthropic configuration
from IPython.display import Markdown, display
import uuid

# Note: deep_researcher is already compiled from the library
# For this demo, we'll use it directly without additional checkpointing
graph = deep_researcher

print("✓ Graph ready for execution")
print("  (Note: The graph is pre-compiled from the library)")

✓ Graph ready for execution
  (Note: The graph is pre-compiled from the library)


### Configuration for Anthropic

We'll configure the system to use:
- **Claude Sonnet 4** for all research, supervision, and report generation
- **Tavily** for web search (you can also use Anthropic's native search)
- **Moderate parallelism** (1 concurrent researcher for cost control)
- **Clarification enabled** (will ask if research scope is unclear)

In [16]:
# Configure for Anthropic with moderate settings
config = {
    "configurable": {
        # Model configuration - using Claude Sonnet 4 for everything
        "research_model": "openai:gpt-4o",
        "research_model_max_tokens": 10000,
        
        "compression_model": "openai:gpt-4o",
        "compression_model_max_tokens": 8192,
        
        "final_report_model": "openai:gpt-4o",
        "final_report_model_max_tokens": 10000,
        
        "summarization_model": "openai:gpt-4o",
        "summarization_model_max_tokens": 8192,
        
        # Research behavior
        "allow_clarification": True,
        "max_concurrent_research_units": 1,  # 1 parallel researcher
        "max_researcher_iterations": 2,      # Supervisor can delegate up to 2 times
        "max_react_tool_calls": 3,           # Each researcher can make up to 3 tool calls
        
        # Search configuration
        "search_api": "tavily",  # Using Tavily for web search
        "max_content_length": 50000,
        
        # Thread ID for this conversation
        "thread_id": str(uuid.uuid4())
    }
}

print("✓ Configuration ready")
print(f"  - Research Model: OpenAI GPT-4o")
print(f"  - Max Concurrent Researchers: 1")
print(f"  - Max Iterations: 2")
print(f"  - Search API: Tavily")

✓ Configuration ready
  - Research Model: OpenAI GPT-4o
  - Max Concurrent Researchers: 1
  - Max Iterations: 2
  - Search API: Tavily


### Execute the Wellness Research

Now let's run the research! We'll ask the system to research evidence-based strategies for improving sleep quality.

The workflow will:
1. **Clarify** - Check if the request is clear (may skip if obvious)
2. **Research Brief** - Transform our request into a structured brief
3. **Supervisor** - Plan research strategy and delegate to researchers
4. **Parallel Research** - Researchers gather information simultaneously
5. **Compression** - Each researcher synthesizes their findings
6. **Final Report** - All findings combined into comprehensive report

In [17]:
# Create our wellness research request
research_request = """
I want to improve my sleep quality. I currently:
- Go to bed at inconsistent times (10pm-1am)
- Use my phone in bed
- Often feel tired in the morning

Please research the best evidence-based strategies for improving sleep quality and create a comprehensive sleep improvement plan for me.
"""

# Execute the graph
async def run_research():
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)

# Run the research
await run_research()

Starting research workflow...


Node: clarify_with_user

Thank you for providing the information about your current sleep challenges and goals. Based on your input—going to bed at inconsistent times, phone usage in bed, and morning fatigue—I have sufficient details to begin researching evidence-based strategies for enhancing sleep quality. I'll now start creating a comprehensive sleep improvement plan tailored to your needs.

Node: write_research_brief

Research Brief Generated:
What are the most effective evidence-based strategies and interventions to improve sleep quality for an individual experiencing inconsistent bedtimes (ranging between 10pm and 1am), habitual phone usage in bed, and persistent morning fatigue, and how can these strategies be integrated into a personalized sleep improvement plan? 

Considerations should include:
- Behavioral changes, such as establishing a consistent sleep schedule.
- Technological interventions, including minimizing screen time.
...

Node: resea

# Comprehensive Sleep Improvement Plan

## Introduction
Improving sleep quality for individuals experiencing inconsistent bedtimes, habitual phone usage in bed, and persistent morning fatigue requires a multi-faceted approach. This comprehensive sleep improvement plan will integrate evidence-based strategies focusing on behavioral changes, technological interventions, environmental adjustments, and potentially medical insights for underlying sleep issues. It also emphasizes flexibility, allowing for adaptation based on individual efficacy.

## Behavioral Changes

### Establishing a Consistent Sleep Schedule
- **Regular Sleep Routine**: Establishing a consistent sleep routine is crucial for long-term sleep quality. A regular bedtime and wake time, even on weekends, aligns the body's internal clock, improving sleep quality and health outcomes such as enhanced mental and cardiovascular health[1][2].
- **Use of Reminders and Alarms**: Set reminders and use alarms to Prompt bedtime preparation and adherence. This helps in avoiding late-night engagements or distractions that hinder a regular sleep schedule[3].

### Sleep Hygiene Practices
- **Mindful Dietary Choices**: Limit caffeine and alcohol consumption, especially in the late afternoon and evening, as they can affect sleep latency and quality[1].
- **Physical Activity**: Incorporate regular physical activity but avoid exercising close to bedtime to prevent overstimulation[3][4].
- **Wind-down Routine**: Develop a calming bedtime routine involving activities like reading, meditation, or journaling. This can help in winding down and signaling the brain that it’s time to sleep[5][6].

## Technological Interventions

### Minimizing Screen Time
- **Reduce Phone Usage in Bed**: Limit the use of phones and other digital screens at least an hour before bedtime. The blue light emitted from these devices interferes with the production of melatonin, the sleep hormone, making it difficult to fall asleep[6][7].
- **Blue Light Filters and Night Mode**: Activate blue light filters or use night mode settings on devices to minimize blue light exposure during evening hours[8].

### Digital Well-Being Tools
- **Screen Time Management Apps**: Utilize apps designed to monitor and limit screen time. These can help in controlling and reducing unnecessary phone usage before bed[9].

## Environmental Adjustments

### Optimizing Sleep Environment
- **Temperature and Light**: Maintain a sleep-conducive environment by ensuring the room is cool, dark, and quiet. Consider blackout curtains and noise-canceling devices if necessary[3].
- **Comfortable Bedding**: Invest in comfortable mattresses and pillows tailored to individual preferences to enhance sleep quality[10].

### Sleep Environment Enhancements
- **Aromatic Ambiance**: Use soothing scents such as lavender or chamomile to create a relaxing ambiance conducive to sleep[11].

## Medical and Therapeutic Insights

### Screening for Sleep Disorders
- **Professional Consultation**: If sleep difficulties persist despite lifestyle changes, consider consulting a sleep specialist to screen for disorders like sleep apnea or insomnia[12]. Medical treatments or therapies may be recommended for underlying conditions.

## Flexibility and Adaptation
- **Iterative Adjustments**: Regularly assess the efficacy of implemented strategies. Be prepared to make necessary adjustments based on what works best for the individual's lifestyle and preferences.

### Personalized Sleep Log
- **Keep a Sleep Diary**: Tracking sleep patterns, bedtime routines, and wake times can help identify trends and areas for improvement. This documentation supports making informed adjustments to the sleep plan[13].

### Reevaluation and Feedback
- **Regular Review**: Schedule periodic reviews of the sleep improvement plan to reflect on progress and make informed changes based on feedback and results.

### Sources
1. [Sleep Physiology and Hygiene](https://pubmed.ncbi.nlm.nih.gov/36841492/)
2. [Consistency in Sleep and Health](https://www.nytimes.com/2026/01/05/well/health-benefits-sleep-consistency.html)
3. [National Sleep Foundation on Sleep Schedule](https://www.thensf.org/setting-a-regular-sleep-schedule/)
4. [Mayo Clinic Sleep Tips](https://www.mayoclinic.org/healthy-lifestyle/adult-health/in-depth/sleep/art-20048379)
5. [Bedtime Routines and Sleep Outcomes](https://www.heart.org/en/healthy-living/healthy-lifestyle/sleep/how-to-sleep-better-with-a-bedtime-routine)
6. [Improving Sleep Routine](https://www.sleepfoundation.org/sleep-hygiene/bedtime-routine-for-adults)
7. [Reducing Blue Light Exposure](https://www.psu.edu/news/health-and-human-development/story/consistent-bedtime-linked-better-child-emotion-and-behavior)
8. [Blue Light and Sleep](https://www.sleepfoundation.org/bedroom-environment/blue-light)
9. [Digital Well-Being and Screen Management](https://pmc.ncbi.nlm.nih.gov/articles/PMC11417809/)
10. [Creating a Comfortable Sleep Environment](https://aquila.usm.edu/cgi/viewcontent.cgi?article=2964&context=dissertations)
11. [Aromatic Influence on Sleep](https://www.sciencedirect.com/science/article/pii/S0163638325000013)
12. [Medical Insights for Sleep Disorders](https://academic.oup.com/sleep/article/47/1/zsad285/7344663)
13. [Effective Sleep Log Practices](https://www.sleephealthjournal.org/article/S2352-7218(23)00166-3/fulltext)


Research workflow completed!


## Task 9: Understanding the Output

Let's break down what happened:

### Phase 1: Clarification
The system checked if your request was clear. Since you provided specific details about your sleep issues, it likely proceeded without asking clarifying questions.

### Phase 2: Research Brief
Your request was transformed into a detailed research brief that guides the supervisor's delegation strategy.

### Phase 3: Supervisor Delegation
The supervisor analyzed the brief and decided how to break down the research:
- Used `think_tool` to plan strategy
- Called `ConductResearch` to delegate to researchers
- Each delegation specified a focused research topic (e.g., sleep hygiene, circadian rhythm, blue light effects)

### Phase 4: Parallel Research
Researchers worked on their assigned topics:
- Each researcher used web search tools to gather information
- Used `think_tool` to reflect after each search
- Decided when they had enough information
- Compressed their findings into clean summaries

### Phase 5: Final Report
All research findings were synthesized into a comprehensive sleep improvement plan with:
- Well-structured sections
- Evidence-based recommendations
- Practical action items
- Sources for further reading

## Task 10: Key Takeaways & Next Steps

### Architecture Benefits
1. **Dynamic Decomposition** - Research structure emerges from the question, not predefined
2. **Parallel Efficiency** - Multiple researchers work simultaneously
3. **ReAct Quality** - Strategic reflection improves search decisions
4. **Scalability** - Handles token limits gracefully through compression
5. **Flexibility** - Easy to add new tools and capabilities

### When to Use This Pattern
- **Complex research questions** that need multi-angle investigation
- **Comparison tasks** where parallel research on different topics is beneficial
- **Open-ended exploration** where structure should emerge dynamically
- **Time-sensitive research** where parallel execution speeds up results

### When to Use Section-Based Instead
- **Highly structured reports** with predefined format requirements
- **Template-based content** where sections are always the same
- **Sequential dependencies** where later sections depend on earlier ones
- **Budget constraints** where token efficiency is critical

### Extend the System
1. **Add MCP Tools** - Integrate specialized tools for your domain
2. **Custom Prompts** - Modify prompts for specific research types
3. **Different Models** - Try different Claude versions or mix models
4. **Persistence** - Use a real database for checkpointing instead of memory

### Learn More
- [LangGraph Documentation](https://langchain-ai.github.io/langgraph/)
- [Open Deep Research Repo](https://github.com/langchain-ai/open_deep_research)
- [Anthropic Claude Documentation](https://docs.anthropic.com/)
- [Tavily Search API](https://tavily.com/)

## ❓ Question #3:

What are the trade-offs of using parallel researchers vs. sequential research? When might you choose one approach over the other?

##### Answer:
Parallel research increases speed and coverage. It works well when the task can be cleanly divided into independent subtopics. It reduces latency and can surface diverse perspectives quickly. However, it increases cost (multiple agents running at once), coordination complexity, and the risk of redundant or inconsistent findings.
Sequential research is slower but more controlled and cost-efficient. It allows each step to build on previous findings, refine the direction, and avoid unnecessary work. It’s better for exploratory or ambiguous tasks where later steps depend on earlier discoveries.
We would choose parallel research for clearly separable, time-sensitive tasks with sufficient budget and sequential research when the problem is evolving, tightly coupled, or when cost and precision matter more than speed.

## ❓ Question #4:

How would you adapt this deep research architecture for a production wellness application? What additional components would you need?

##### Answer:
I would adapt this by securing persistant storage (DB for profiles/metrics), adding a strict safety layer with authetification methods, adding monitoring with observabillity (structured logs, tracing, ...), adding a lot of tests as well as the CI/CD & deployment controls.

## 🏗️ Activity #2: Custom Wellness Research

Using what you've learned, run a custom wellness research task.

**Requirements:**
1. Create a wellness-related research question (exercise, nutrition, stress, etc.)
2. Modify the configuration for your use case
3. Run the research and analyze the output
4. Document what worked well and what could be improved

**Experiment ideas:**
- Research exercise routines for specific conditions (bad knee, lower back pain)
- Compare different stress management techniques
- Investigate nutrition strategies for specific goals
- Explore meditation and mindfulness research

**YOUR CODE HERE**

In [18]:
# YOUR CODE HERE
# Create your own wellness research request and run it

my_wellness_request = """
Compare three evidence-based morning exercise routines for people with chronic lower-back pain.
For each routine:
- Provide the routine steps and total time.
- Summarize supporting evidence (cite studies / reviews).
- Describe risks and safety modifications for severe pain or limited mobility.
- Give a short recommendation (who should/shouldn't use it).
"""

# Optionally modify the config
my_config = {
    "configurable": {
        "research_model": "openai:gpt-4o",
        "research_model_max_tokens": 10000,
        "compression_model": "openai:gpt-4o",
        "compression_model_max_tokens": 8192,
        "final_report_model": "openai:gpt-4o",
        "final_report_model_max_tokens": 10000,
        "summarization_model": "openai:gpt-4o",
        "summarization_model_max_tokens": 8192,
        "allow_clarification": True,
        "max_concurrent_research_units": 1,
        "max_researcher_iterations": 2,
        "max_react_tool_calls": 3,
        "search_api": "tavily",
        "max_content_length": 50000,
        "thread_id": str(uuid.uuid4())
    }
}

# Run your research
# await run_custom_research(my_wellness_request, my_config)

async def run_custom_research(research_request, config):
    """Run the research workflow and display results."""
    print("Starting research workflow...\n")
    
    async for event in graph.astream(
        {"messages": [{"role": "user", "content": research_request}]},
        config,
        stream_mode="updates"
    ):
        # Display each step
        for node_name, node_output in event.items():
            print(f"\n{'='*60}")
            print(f"Node: {node_name}")
            print(f"{'='*60}")
            
            if node_name == "clarify_with_user":
                if "messages" in node_output:
                    last_msg = node_output["messages"][-1]
                    print(f"\n{last_msg.content}")
            
            elif node_name == "write_research_brief":
                if "research_brief" in node_output:
                    print(f"\nResearch Brief Generated:")
                    print(f"{node_output['research_brief'][:500]}...")
            
            elif node_name == "supervisor":
                print(f"\nSupervisor planning research strategy...")
                if "supervisor_messages" in node_output:
                    last_msg = node_output["supervisor_messages"][-1]
                    if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                        print(f"Tool calls: {len(last_msg.tool_calls)}")
                        for tc in last_msg.tool_calls:
                            print(f"  - {tc['name']}")
            
            elif node_name == "supervisor_tools":
                print(f"\nExecuting supervisor's tool calls...")
                if "notes" in node_output:
                    print(f"Research notes collected: {len(node_output['notes'])}")
            
            elif node_name == "final_report_generation":
                if "final_report" in node_output:
                    print(f"\n" + "="*60)
                    print("FINAL REPORT GENERATED")
                    print("="*60 + "\n")
                    display(Markdown(node_output["final_report"]))
    
    print("\n" + "="*60)
    print("Research workflow completed!")
    print("="*60)


results = await run_custom_research(my_wellness_request, my_config)

Starting research workflow...


Node: clarify_with_user

The information provided is sufficient to proceed with the research. We will compare three evidence-based morning exercise routines specifically tailored for individuals with chronic lower-back pain. Each routine will detail the steps involved, total time required, supporting scientific evidence, safety modifications for severe pain or limited mobility, and recommendations for specific user groups. We will now begin the research process to deliver a comprehensive report tailored to your needs.

Node: write_research_brief

Research Brief Generated:
How do three different evidence-based morning exercise routines tailored for individuals with chronic lower-back pain compare in terms of effectiveness, safety, and user suitability? Specifically, I need information on each routine's detailed steps, total time required, scientific support from studies or reviews, safety modifications for individuals with severe pain or limited mobility,

# Comparison of Evidence-Based Morning Exercise Routines for Chronic Lower Back Pain

## Introduction

Chronic lower back pain is a prevalent issue, often impacting the quality of life and mobility of individuals worldwide. Morning exercise routines designed for this condition aim to alleviate pain, improve mobility, and enhance overall well-being. This report explores three distinct evidence-based morning exercise routines tailored for individuals with chronic lower back pain, comparing their effectiveness, safety, and suitability for users with varying levels of pain and mobility. The routines include the McKenzie method, Pilates, and Yoga, each backed by scientific research and guidelines.

## Routine 1: McKenzie Method

### Routine Steps and Total Time
The McKenzie method, developed by physiotherapist Robin McKenzie, focuses on exercises that extend the spine to centralize pain to the lower back. Key exercises include:

- **Prone Lying**: Lie face down on a mat, resting for 2-3 minutes to allow the low back to relax.
- **Prone Extension (Press-Ups)**: From a prone position, push up with arms while keeping hips on the ground. Hold for 2-3 seconds, repeat 10 times.
- **Standing Extension**: Stand with feet shoulder-width apart, place hands on lower back, and lean backwards gently. Repeat 10 times.

The typical routine can be completed within 10-15 minutes[1][2].

### Supporting Evidence
Numerous studies support the McKenzie method for chronic lower back pain management. It has shown significant pain reduction and improved functional outcomes compared to other therapies. The method is effective in centralizing pain and is recommended for conditions where mechanical pain is present without severe underlying pathology[1][3].

### Safety Modifications
Modifications are crucial for individuals with severe pain or limited mobility. Exercises can be customized, such as reducing the range of movement or performing exercises under professional supervision. Those with acute back injuries or serious spinal conditions are advised to avoid this method[4][5].

### Suitability and Recommendations
The McKenzie method is suitable for individuals with chronic mechanical back pain but not recommended for those with underlying conditions like tumors or in immediate post-surgical recovery[3][5].

## Routine 2: Pilates

### Routine Steps and Total Time
Pilates focuses on core strengthening and stability, which supports the lower back. A basic morning routine includes:

- **Pelvic Tilts**: Lie on your back with knees bent, tilt pelvis to flatten back on the mat, hold for a few seconds. Repeat 10 times.
- **Dead Bug**: Lying on your back, lift legs and arms, then lower opposite arm and leg while maintaining a stable core. Repeat 10 times on each side.
- **Bridges**: Lie on back, lift hips towards the ceiling, squeezing glutes. Hold for 2 seconds, repeat 10 times.

This routine typically takes 15-20 minutes[6][7].

### Supporting Evidence
Pilates has been supported by various studies for its effectiveness in reducing chronic lower back pain. The emphasis on core muscles leads to decreased pain levels and improved functional movement. Research suggests it is particularly effective when incorporated long-term[6][8].

### Safety Modifications
For individuals with severe pain or mobility limitations, exercises can be adjusted by reducing the intensity or range. Consultation with a professional is recommended for tailored modifications[9].

### Suitability and Recommendations
Pilates is suitable for individuals seeking low-impact exercise focusing on core strength. It is not ideal for individuals without proper supervision if they lack prior experience, particularly those with severe pain[8][9].

## Routine 3: Yoga

### Routine Steps and Total Time
Yoga incorporates stretching, strength, and flexibility exercises suitable for lower back pain, such as:

- **Cat-Cow Stretch**: On all fours, alternate between arching back upwards and downwards. Repeat 10 times.
- **Child’s Pose**: Sit back on heels with arms stretched forward, hold for 2-3 minutes.
- **Supine Spinal Twist**: Lying on your back, gently twist with one knee across the body. Hold for 30 seconds on each side.

A morning yoga routine can be effectively completed in 20-30 minutes[10][11].

### Supporting Evidence
Yoga is extensively researched and has shown positive effects on chronic lower back pain by reducing pain and enhancing mental well-being. Its comprehensive approach addresses both physical and psychological aspects of pain[10][12].

### Safety Modifications
Safety measures include using props and modifying poses to individual capabilities. People with severe pain should start with gentle poses and consult a certified yoga instructor for guidance[13].

### Suitability and Recommendations
Yoga is widely recommended for its holistic benefits but should be approached cautiously by those with severe pain. It may not be suitable for individuals who are unable to maintain balance or have acute musculoskeletal injuries without guidance.

## Comparison and Conclusion

Each of the three routines offers distinct benefits for managing chronic lower back pain. The McKenzie method is particularly effective for mechanical pain, while Pilates strengthens the core, supporting the back. Yoga combines physical and psychological benefits but may require adaptations for those with severe limitations. Selecting a routine depends on individual pain levels, mobility, personal preferences, and medical advice. Consultation with healthcare professionals is recommended to tailor an exercise plan that aligns with personal health needs.

### Sources

1. [7 McKenzie Method Exercises for Back Pain and Sciatica](https://www.spine-health.com/wellness/exercise/7-mckenzie-method-exercises-back-pain-and-sciatica)  
2. [McKenzie Exercises for Low Back Pain](https://www.verywellhealth.com/mckenzie-exercises-for-your-low-back-2696222)  
3. [Effectiveness of McKenzie Method of Mechanical Diagnosis and Therapy](https://www.jospt.org/doi/10.2519/jospt.2018.7562)  
4. [When to Avoid McKenzie Method Exercises](https://www.spine-health.com/blog/when-avoid-mckenzie-method-exercises-5-potential-risks)  
5. [Modifying The McKenzie Method](https://www.physio-network.com/blog/mckenzie-method/)  
6. [Pilates for Rehabilitation of Chronic Lower Back Pain](https://pubmed.ncbi.nlm.nih.gov/28345466/)  
7. [Introduction to Pilates](https://www.acefitness.org/education-and-resources/lifestyle/blog/6599/pilates-exercises-for-the-core/)  
8. [The effectiveness of Pilates exercise in the treatment of chronic lower back pain](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC3184497/)  
9. [Pilates for Chronic Lower Back Pain](https://www.physicianassistantforum.com/)  
10. [Effect of Yoga in the Management of Chronic Pain](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC9350964/)  
11. [Yoga Exercises for Back Pain](https://www.yogajournal.com)  
12. [Yoga for Chronic Back Pain: A Scientific Review](https://www.frontiersin.org/articles/10.3389/fpsyt.2020.573252/full)  
13. [Guide to Yoga for people with lower back pain](https://www.yogabasics.com)  


Research workflow completed!
